# Build silver dimensions

This converts customer, location, contact, employee, sales reps, and technicians into Silver delta tables.

In [6]:
# Attach the schema-enabled Lakehouse containing the Bronze shortcuts and
# make it the default Lakehouse before running this notebook.
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
METADATA_SCHEMA = "metadata"
COMPANY_KEY = "NAC"  # Change when processing another GP company database.

def q(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"

def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"

def table_exists(schema_name: str, table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(f"{schema_name}.{table_name}")
    except Exception:
        return False

def require_tables(table_names: list[str], schema_name: str = BRONZE_SCHEMA) -> None:
    missing = [name for name in table_names if not table_exists(schema_name, name)]
    if missing:
        raise RuntimeError(
            f"Missing tables in {schema_name}: {', '.join(missing)}. "
            "Confirm the OneLake shortcuts and exact table names."
        )

company = sql_string(COMPANY_KEY)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(SILVER_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(GOLD_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(METADATA_SCHEMA)}")
print(f"Company: {COMPANY_KEY}; Bronze schema: {BRONZE_SCHEMA}")

StatementMeta(, 583ca52f-54f0-4594-bf31-c6a4cef1c05a, 8, Finished, Available, Finished, False)

Company: NAC; Bronze schema: bronze


## Customer and location tables
These use the customer (SV00100) and location (SV00200) master tables.

In [7]:
require_tables(["SV00100", "SV00200"])

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.customer
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(c.CUSTNMBR) AS customer_code,
    TRIM(l.CUSTNAME) AS customer_name,
    TRIM(l.ADDRESS1) AS customer_address1,
    TRIM(l.CITY) AS customer_city,
    TRIM(l.STATE) AS customer_state,
    TRIM(l.ZIP) AS customer_zip,
    NULLIF(TRIM(l.PHONE1), '') AS customer_phone1,
    NULLIF(TRIM(l.FAX), '') AS customer_fax,
    NULLIF(TRIM(l.CNTCPRSN), '') AS customer_contact,
    CASE WHEN l.WS_Latitude BETWEEN 24 AND 51
         THEN CAST(l.WS_Latitude AS DOUBLE) END AS latitude_north,
    CASE WHEN l.WS_Longitude BETWEEN 69 AND 126
         THEN -ABS(CAST(l.WS_Longitude AS DOUBLE)) END AS longitude_west,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00100 c
LEFT JOIN {q(BRONZE_SCHEMA)}.SV00200 l
  ON TRIM(c.CUSTNMBR) = TRIM(l.CUSTNMBR)
 AND TRIM(l.ADRSCODE) = 'MAIN'
WHERE TRIM(c.CUSTNMBR) <> ''
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.location
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(CUSTNMBR) AS customer_code,
    TRIM(ADRSCODE) AS location_code,
    TRIM(CUSTNAME) AS customer_name,
    TRIM(LOCATNNM) AS location_name,
    TRIM(ADDRESS1) AS location_address1,
    CAST(NULL AS STRING) AS location_address2,
    TRIM(CITY) AS location_city,
    TRIM(STATE) AS location_state,
    TRIM(ZIP) AS location_zip,
    TRIM(SLPRSNID) AS sales_rep_code,
    SUBSTRING(TRIM(PHONE1), 1, 10) AS site_phone1,
    SUBSTRING(TRIM(PHONE2), 1, 10) AS site_phone2,
    CONCAT(TRIM(CUSTNMBR), TRIM(ADRSCODE)) AS location_xref,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00200
WHERE TRIM(CUSTNMBR) <> '' AND TRIM(ADRSCODE) <> ''
""")

StatementMeta(, 583ca52f-54f0-4594-bf31-c6a4cef1c05a, 9, Finished, Available, Finished, False)

DataFrame[]

## Contacts
This uses tables SV01100 and SV00205.

In [8]:
require_tables(["SV01100", "SV00205"])

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.contact
USING DELTA
AS
SELECT
    {company} AS company_key,
    CAST(l.Contact_ID AS STRING) AS contact_code,
    TRIM(l.CUSTNMBR) AS customer_code,
    TRIM(l.ADRSCODE) AS location_code,
    TRIM(c.WS_Contact_Name) AS contact_name,
    LOWER(TRIM(c.E_Mail_Address)) AS contact_email,
    TRIM(c.ADDRESS1) AS contact_address1,
    TRIM(c.ADDRESS2) AS contact_address2,
    TRIM(c.CITY) AS contact_city,
    TRIM(c.STATE) AS contact_state,
    TRIM(c.Postal_Code) AS contact_zip,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV01100 c
LEFT JOIN {q(BRONZE_SCHEMA)}.SV00205 l
  ON c.Contact_ID = l.Contact_ID
""")

StatementMeta(, 583ca52f-54f0-4594-bf31-c6a4cef1c05a, 10, Finished, Available, Finished, False)

DataFrame[]

## Employees, sales reps, and technicians
This uses tables UPR00100, UPR40300, SY01200, RM00301, and SV00115.

In [9]:
require_tables(["UPR00100", "UPR40300", "SY01200", "RM00301", "SV00115"])

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.employee
USING DELTA
AS
WITH email AS (
    SELECT
        TRIM(Master_ID) AS employee_code_text,
        LOWER(TRIM(INET1)) AS employee_email,
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(Master_ID)
            ORDER BY TRIM(INET1) DESC
        ) AS rn
    FROM {q(BRONZE_SCHEMA)}.SY01200
    WHERE Master_Type = 'EMP'
)
SELECT
    {company} AS company_key,
    TRY_CAST(emp.EMPLOYID AS BIGINT) AS employee_code,
    TRIM(emp.LASTNAME) AS employee_last_name,
    TRIM(emp.FRSTNAME) AS employee_first_name,
    TRIM(emp.JOBTITLE) AS employee_job_code,
    e.employee_email,
    TRIM(emp.DEPRTMNT) AS department_code,
    TRIM(dep.DSCRIPTN) AS department_name,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.UPR00100 emp
LEFT JOIN {q(BRONZE_SCHEMA)}.UPR40300 dep
  ON emp.DEPRTMNT = dep.DEPRTMNT
LEFT JOIN email e
  ON TRIM(emp.EMPLOYID) = e.employee_code_text AND e.rn = 1
WHERE emp.INACTIVE = 0
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.employee_sales_rep
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRY_CAST(EMPLOYID AS BIGINT) AS employee_code,
    TRIM(SLPRSNID) AS sales_rep_code,
    TRIM(SLPRSNFN) AS employee_first_name,
    TRIM(SPRSNSLN) AS employee_last_name,
    TRIM(Phone1) AS employee_phone,
    CAST(INACTIVE AS INT) AS inactive_flag,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.RM00301
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.technician
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(t.Technician_ID) AS technician_username,
    TRIM(t.Technician_Team) AS technician_team,
    TRIM(t.Technician_Long_Name) AS technician_name,
    TRY_CAST(t.Employid AS BIGINT) AS employee_code,
    LOWER(TRIM(e.INET1)) AS employee_email,
    CAST(t.SV_Inactive AS INT) AS inactive_flag,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00115 t
LEFT JOIN {q(BRONZE_SCHEMA)}.SY01200 e
  ON TRIM(CAST(t.EMPLOYID AS STRING)) = TRIM(e.Master_ID)
 AND e.Master_Type = 'EMP'
WHERE t.Employid <> 0
""")

if table_exists(BRONZE_SCHEMA, "Z_Technician_teams"):
    spark.sql(f"""
    CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.technician_team_assignment
    USING DELTA
    AS
    SELECT
        t.company_key, t.employee_code, t.technician_username,
        t.technician_name, t.employee_email, t.technician_team,
        TRIM(m.TeamLead_name) AS team_lead_name,
        LOWER(TRIM(m.TeamLead_email)) AS team_lead_email,
        TRIM(m.Dispatcher_name) AS dispatcher_name,
        LOWER(TRIM(m.Dispatcher_email)) AS dispatcher_email,
        CURRENT_TIMESTAMP() AS _processed_at
    FROM {q(SILVER_SCHEMA)}.technician t
    LEFT JOIN {q(BRONZE_SCHEMA)}.Z_Technician_teams m
      ON t.technician_team = TRIM(m.Team)
    WHERE t.inactive_flag = 0
    """)
else:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.technician_team_assignment
    USING DELTA
    AS
    SELECT
        company_key, employee_code, technician_username, technician_name,
        employee_email, technician_team,
        CAST(NULL AS STRING) AS team_lead_name,
        CAST(NULL AS STRING) AS team_lead_email,
        CAST(NULL AS STRING) AS dispatcher_name,
        CAST(NULL AS STRING) AS dispatcher_email,
        CURRENT_TIMESTAMP() AS _processed_at
    FROM {q(SILVER_SCHEMA)}.technician
    WHERE inactive_flag = 0
    """)
    print("bronze.Z_Technician_teams was not found; team-lead and dispatcher fields are null.")

StatementMeta(, 583ca52f-54f0-4594-bf31-c6a4cef1c05a, 11, Finished, Available, Finished, False)

bronze.Z_Technician_teams was not found; team-lead and dispatcher fields are null.


## Review

In [10]:
for name in ["customer", "location", "contact", "employee", "employee_sales_rep",
             "technician", "technician_team_assignment"]:
    count = spark.table(f"{SILVER_SCHEMA}.{name}").count()
    print(f"{SILVER_SCHEMA}.{name}: {count:,} rows")

StatementMeta(, 583ca52f-54f0-4594-bf31-c6a4cef1c05a, 12, Finished, Available, Finished, False)

silver.customer: 1,002 rows
silver.location: 3,386 rows
silver.contact: 3,296 rows
silver.employee: 113 rows
silver.employee_sales_rep: 38 rows
silver.technician: 144 rows
silver.technician_team_assignment: 88 rows
